In [17]:
import pandas as pd

In [18]:
df=pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

In [19]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

# --- CLEANING PHASE ---

In [21]:
# Droping unnecessary or sensitive columns
useless_cols = [
    'Product Description', 'Customer Email', 'Customer Password', 
    'Product Image', 'Customer Fname', 'Customer Lname', 'Customer Street',
    'Order Zipcode', 'Product Status'
]
df.drop(columns=useless_cols, inplace=True, errors='ignore')

In [22]:
# Handeling missing values
df['Customer Zipcode'] = df['Customer Zipcode'].fillna(0)

In [23]:
# Converting the date columns to proper datetime format
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 44 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Type                           180519 non-null  object        
 1   Days for shipping (real)       180519 non-null  int64         
 2   Days for shipment (scheduled)  180519 non-null  int64         
 3   Benefit per order              180519 non-null  float64       
 4   Sales per customer             180519 non-null  float64       
 5   Delivery Status                180519 non-null  object        
 6   Late_delivery_risk             180519 non-null  int64         
 7   Category Id                    180519 non-null  int64         
 8   Category Name                  180519 non-null  object        
 9   Customer City                  180519 non-null  object        
 10  Customer Country               180519 non-null  object        
 11  

# --- FEATURE ENGINEERING---

In [25]:
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day_name()

In [26]:
df['shipping_gap'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']

In [27]:
df['late_flag'] = (df['Days for shipping (real)'] > df['Days for shipment (scheduled)']).astype(int)

In [28]:
df['profit_margin'] = df['Order Profit Per Order'] / df['Sales']

# --- DATA FORMATTING ---

In [29]:
df.rename(columns={
    'order date (DateOrders)': 'order_date',
    'shipping date (DateOrders)': 'shipping_date',
    'Days for shipping (real)': 'actual_shipping_days',
    'Days for shipment (scheduled)': 'scheduled_shipping_days',
    'Order Item Total': 'net_sales',
    'Order Profit Per Order': 'total_profit',
    'Order Region': 'region'
}, inplace=True)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Type                      180519 non-null  object        
 1   actual_shipping_days      180519 non-null  int64         
 2   scheduled_shipping_days   180519 non-null  int64         
 3   Benefit per order         180519 non-null  float64       
 4   Sales per customer        180519 non-null  float64       
 5   Delivery Status           180519 non-null  object        
 6   Late_delivery_risk        180519 non-null  int64         
 7   Category Id               180519 non-null  int64         
 8   Category Name             180519 non-null  object        
 9   Customer City             180519 non-null  object        
 10  Customer Country          180519 non-null  object        
 11  Customer Id               180519 non-null  int64         
 12  Cu

In [32]:
df.to_csv('Cleaned_SupplyChain_Data.csv', index=False)